# 📊 Data Documentation — Pump Failure Prediction MVP

## 1. Data Sources

| Property | Details |
|---|---|
| **Source** | Laboratory test bench — Silesian University of Technology, Poland |
| **Reference** | Rojek & Blachnik (2024) — DOI: 10.20944/preprints202407.0284.v1 |
| **Public repository** | [Kaggle Dataset](https://www.kaggle.com/datasets/mbjunior/valve-plate-failure-prediction-in-hydraulic-pumps) |
| **Format** | 4 CSV files, one per operational state |
| **Volume** | ~153,000 samples total |
| **Time resolution** | 1 sample/second (averaged readings) |
| **Update frequency** | Static dataset (MVP) — future: real-time OT integration |

### Raw Files

| File | Label | Class | Samples |
|---|---|---|---|
| `dane_OT.csv` | 0 | Normal | ~68,347 |
| `dane_ut1.csv` | 1 | Valve Plate Wear | ~54,053 |
| `dane_ut2.csv` | 2 | Simulated Failure 1 | ~14,333 |
| `dane_ut3.csv` | 3 | Simulated Failure 2 | ~16,472 |

---

## 2. Data Dictionary

### 2.1 Raw Features (7 sensor readings)

| Feature | Unit | Sensor ID | Description | Position |
|---|---|---|---|---|
| `Pressure - leak line` | MPa | P17-AI05 | Hydraulic pressure at leak line | Return circuit |
| `Temperature - leak line` | °C | T13-AI06 | Fluid temperature at leak line | Return circuit |
| `Pressure - output` | MPa | P19-AI01 | Pump output pressure | Output side |
| `Temperature - output` | °C | T06-AI13 | Fluid temperature at output | Output side |
| `Flow - leak line` | L/min | STAUFF-AI19 | Volumetric flow at leak line | Return circuit |
| `Flow - output` | L/min | DZR-AI08 | Pump output volumetric flow | Output side |
| `Temp. diff` | °C | Derived | Temperature differential (output - suction) | Derived |

### 2.2 Removed Features

| Feature | Reason |
|---|---|
| `Czas` | Timestamp string — not a physical state feature |
| `Czas2` | Timestamp string — not a physical state feature |
| `Temperature - suction line` | Correlation = 0.992 with `Temperature - output` → redundant |
| `stan` | Binary normal/anomaly — subset of `label`, adds no information |
| `label_name` | Text label — redundant with numeric `label` |

### 2.3 Target Variable

| Column | Type | Values |
|---|---|---|
| `label` | Integer (0–3) | 0=Normal, 1=Wear, 2=SF1, 3=SF2 |
| `label_name` | String | Human-readable class name |

---

## 3. Exploratory Data Analysis (EDA)

### 3.1 Class Distribution

| Class | Samples | % of Total | Imbalance Note |
|---|---|---|---|
| Normal | 68,347 | 44.6% | Majority class |
| Valve Plate Wear | 54,053 | 35.3% | Moderate |
| Simulated Failure 1 | 14,333 | 9.4% | Minority |
| Simulated Failure 2 | 16,472 | 10.8% | Minority |

**Imbalance treatment:** Borderline-SMOTE applied on training set only.

### 3.2 Missing Values

| Feature | Missing Count | % Missing | Treatment |
|---|---|---|---|
| `Flow - output` | 6,846 | 4.47% | Median imputation |
| All others | 0 | 0% | None required |

### 3.3 Outliers

| Feature | Outliers (Z>3σ) | % | Treatment |
|---|---|---|---|
| `Pressure - leak line` | 309 | 0.20% | Clipped to ±3σ |
| All others | 0 | 0% | None required |

### 3.4 Correlation Analysis

| Pair | Correlation | Decision |
|---|---|---|
| Temperature - suction line × Temperature - output | 0.992 | Remove suction line |

### 3.5 Key EDA Findings

- **`Temperature - leak line`** shows stable distribution (35–40°C) in Normal
  state and higher variance (20–40°C) in failure states → strong discriminator
- **`Pressure - leak line`** shows similar patterns across all classes in
  isolation → useful mainly in derived features (ratios, rolling stats)
- **`Flow - leak line`** is the most important individual sensor
  (aligned with paper findings)
- No strong temporal degradation pattern found → confirms ensemble ML
  approach over LSTM for this dataset

---

## 4. Feature Engineering

### 4.1 Engineered Features (30 new features → 37 total)

#### Ratio Features (3)

| Feature | Formula | Physical Meaning |
|---|---|---|
| `ratio_flow_leak_output` | `Flow_leak / (Flow_output + ε)` | Leakage efficiency — increases with valve plate wear |
| `ratio_pressure_leak_output` | `Pressure_leak / (Pressure_output + ε)` | Pressure drop ratio — indicates valve degradation |
| `ratio_temp_diff_output` | `Temp_diff / (Temperature_output + ε)` | Normalized thermal stress |

#### Rolling Statistics (24)

Applied to: `Pressure - leak line`, `Temperature - leak line`,
`Flow - leak line`, `Flow - output`

| Statistic | Windows | Features created |
|---|---|---|
| Rolling Mean | 5s, 10s, 30s | 12 features |
| Rolling Std | 5s, 10s, 30s | 12 features |

> **Rationale:** Rolling std captures variance differences between classes
> identified in EDA (Temperature - leak line behavior).

#### Delta Features (3)

| Feature | Formula | Physical Meaning |
|---|---|---|
| `pressure__output_delta` | `P_out[t] - P_out[t-1]` | Abrupt pressure changes |
| `pressure__leak_line_delta` | `P_leak[t] - P_leak[t-1]` | Abrupt leak pressure changes |
| `flow__output_delta` | `Q_out[t] - Q_out[t-1]` | Abrupt flow changes |

### 4.2 Feature Selection

- **Method:** VarianceThreshold (threshold=0.01)
- **Result:** No features removed — all 37 features have sufficient variance
- **Final feature count:** 37

### 4.3 Normalization

- **Method:** StandardScaler (zero mean, unit variance)
- **Fit:** Training set only (before SMOTE)
- **Apply:** Training, validation, and test sets
- **Saved artifact:** `models/scaler.joblib`